# Clarkson II anomaly investigation

This notebook investigates the unmatched press/release events and negative hold-time pairs found during event-feature extraction. It does not modify raw, normalized, or feature files.

The investigation records:

- Per-user press/release balance and unmatched-event counts
- Keys associated with negative hold-time pairs
- Representative negative-pair examples
- Representative unmatched-release examples
- Representative unmatched-press examples remaining at the end of each user stream

In [1]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import gzip
import json

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / 'clarkson2_preprocessing').is_dir()), HERE)
WORK_DIR = PROJECT_ROOT / 'clarkson2_preprocessing'
NORMALIZED_DIR = WORK_DIR / 'outputs' / 'normalized_events'
OUTPUT_DIR = WORK_DIR / 'outputs' / 'anomaly_investigation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_files = sorted(NORMALIZED_DIR.glob('*.tsv.gz'))
if not input_files:
    raise FileNotFoundError(f'No normalized files found in {NORMALIZED_DIR}')

MAX_EXAMPLES_PER_USER = 20
print('Users to investigate:', len(input_files))
print('Output directory:', OUTPUT_DIR)

Users to investigate: 100
Output directory: C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\clarkson2_preprocessing\outputs\anomaly_investigation


## Investigation logic

The previous feature notebook used a FIFO queue for repeated presses of the same key. This notebook reproduces that policy and saves evidence for suspicious cases.

A negative pair means the release timestamp was earlier than the queued press timestamp. This may indicate a timestamp/order problem, repeated-key ambiguity, or a limitation of the pairing policy; it is not automatically deleted from the investigation.

In [2]:
SUMMARY_FIELDS = [
    'user_id', 'input_rows', 'press_events', 'release_events',
    'paired_rows', 'negative_hold_pairs', 'unmatched_release_events',
    'unmatched_press_events', 'timestamp_backsteps',
    'equal_timestamps', 'negative_hold_rate', 'press_release_ratio'
]
NEGATIVE_FIELDS = [
    'user_id', 'release_event_number', 'press_event_number', 'key',
    'press_timestamp_100ns', 'release_timestamp_100ns',
    'hold_seconds', 'events_between_press_and_release'
]
UNMATCHED_RELEASE_FIELDS = [
    'user_id', 'release_event_number', 'key', 'release_timestamp_100ns'
]
UNMATCHED_PRESS_FIELDS = [
    'user_id', 'press_event_number', 'key', 'press_timestamp_100ns'
]

def investigate_file(input_path):
    user_id = input_path.name.removesuffix('.tsv.gz')
    stats = Counter()
    queues = defaultdict(deque)
    negative_examples = []
    unmatched_release_examples = []
    previous_timestamp = None
    press_event_number = 0
    release_event_number = 0

    with gzip.open(input_path, 'rt', encoding='utf-8', newline='') as source:
        reader = csv.DictReader(source, delimiter='\t')
        for event_number, row in enumerate(reader, start=1):
            stats['input_rows'] += 1
            try:
                timestamp = int(row['timestamp_100ns'])
                flag = row['flag']
                key = row['key']
            except (KeyError, TypeError, ValueError):
                stats['invalid_rows'] += 1
                continue

            if previous_timestamp is not None:
                if timestamp < previous_timestamp:
                    stats['timestamp_backsteps'] += 1
                elif timestamp == previous_timestamp:
                    stats['equal_timestamps'] += 1
            previous_timestamp = timestamp

            if flag == '0':
                press_event_number += 1
                queues[key].append((timestamp, press_event_number, event_number))
                stats['press_events'] += 1
            elif flag == '1':
                release_event_number += 1
                stats['release_events'] += 1
                if not queues[key]:
                    stats['unmatched_release_events'] += 1
                    if len(unmatched_release_examples) < MAX_EXAMPLES_PER_USER:
                        unmatched_release_examples.append({
                            'user_id': user_id,
                            'release_event_number': release_event_number,
                            'key': key,
                            'release_timestamp_100ns': timestamp,
                        })
                else:
                    press_timestamp, source_press_number, source_event_number = queues[key].popleft()
                    hold_seconds = (timestamp - press_timestamp) / 10_000_000
                    if hold_seconds < 0:
                        stats['negative_hold_pairs'] += 1
                        stats[f'negative_key_{key}'] += 1
                        if len(negative_examples) < MAX_EXAMPLES_PER_USER:
                            negative_examples.append({
                                'user_id': user_id,
                                'release_event_number': release_event_number,
                                'press_event_number': source_press_number,
                                'key': key,
                                'press_timestamp_100ns': press_timestamp,
                                'release_timestamp_100ns': timestamp,
                                'hold_seconds': hold_seconds,
                                'events_between_press_and_release': event_number - source_event_number,
                            })
                    else:
                        stats['paired_rows'] += 1

    unmatched_press_examples = []
    for key, queue in queues.items():
        for press_timestamp, source_press_number, source_event_number in list(queue)[:MAX_EXAMPLES_PER_USER - len(unmatched_press_examples)]:
            unmatched_press_examples.append({
                'user_id': user_id,
                'press_event_number': source_press_number,
                'key': key,
                'press_timestamp_100ns': press_timestamp,
            })
        if len(unmatched_press_examples) >= MAX_EXAMPLES_PER_USER:
            break

    stats['unmatched_press_events'] = sum(len(queue) for queue in queues.values())
    stats['user_id'] = user_id
    stats['negative_hold_rate'] = (stats['negative_hold_pairs'] / max(stats['paired_rows'] + stats['negative_hold_pairs'], 1))
    stats['press_release_ratio'] = stats['press_events'] / max(stats['release_events'], 1)
    return dict(stats), negative_examples, unmatched_release_examples, unmatched_press_examples

summaries = []
negative_examples = []
unmatched_release_examples = []
unmatched_press_examples = []
for index, input_path in enumerate(input_files, start=1):
    summary, negative, releases, presses = investigate_file(input_path)
    summaries.append(summary)
    negative_examples.extend(negative)
    unmatched_release_examples.extend(releases)
    unmatched_press_examples.extend(presses)
    if index % 10 == 0 or index == len(input_files):
        print(f'Investigated {index}/{len(input_files)} files')

def write_rows(path, fieldnames, rows):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

write_rows(OUTPUT_DIR / 'anomaly_summary.csv', SUMMARY_FIELDS, summaries)
write_rows(OUTPUT_DIR / 'negative_hold_examples.csv', NEGATIVE_FIELDS, negative_examples)
write_rows(OUTPUT_DIR / 'unmatched_release_examples.csv', UNMATCHED_RELEASE_FIELDS, unmatched_release_examples)
write_rows(OUTPUT_DIR / 'unmatched_press_examples.csv', UNMATCHED_PRESS_FIELDS, unmatched_press_examples)
(OUTPUT_DIR / 'anomaly_summary.json').write_text(json.dumps(summaries, indent=2), encoding='utf-8')

total = Counter()
for row in summaries:
    for key in ['input_rows', 'press_events', 'release_events', 'paired_rows',
                'negative_hold_pairs', 'unmatched_release_events',
                'unmatched_press_events', 'timestamp_backsteps', 'equal_timestamps']:
        total[key] += row.get(key, 0)

print('Negative hold pairs:', total['negative_hold_pairs'])
print('Unmatched releases:', total['unmatched_release_events'])
print('Unmatched presses:', total['unmatched_press_events'])
print('Negative examples saved:', len(negative_examples))
print('Unmatched release examples saved:', len(unmatched_release_examples))
print('Unmatched press examples saved:', len(unmatched_press_examples))

Investigated 10/100 files


Investigated 20/100 files


Investigated 30/100 files


Investigated 40/100 files


Investigated 50/100 files


Investigated 60/100 files


Investigated 70/100 files


Investigated 80/100 files


Investigated 90/100 files


Investigated 100/100 files
Negative hold pairs: 134225
Unmatched releases: 19319
Unmatched presses: 4391358
Negative examples saved: 140
Unmatched release examples saved: 1768
Unmatched press examples saved: 1920


## Highest-anomaly users and keys

The following summaries help distinguish a small number of problematic users from a dataset-wide issue.

In [3]:
top_negative = sorted(summaries, key=lambda row: row.get('negative_hold_pairs', 0), reverse=True)[:15]
top_unmatched = sorted(summaries, key=lambda row: row.get('unmatched_press_events', 0), reverse=True)[:15]

print('Top users by negative hold pairs:')
for row in top_negative:
    print(row['user_id'], row.get('negative_hold_pairs', 0), 'negative pairs;',
          row.get('negative_hold_rate', 0), 'rate')

print('\nTop users by unmatched presses:')
for row in top_unmatched:
    print(row['user_id'], row.get('unmatched_press_events', 0), 'unmatched presses;',
          row.get('press_release_ratio', 0), 'press/release ratio')

key_counts = Counter()
for row in summaries:
    for key, value in row.items():
        if key.startswith('negative_key_'):
            key_counts[key.removeprefix('negative_key_')] += value
print('\nKeys in negative hold pairs:')
for key, count in key_counts.most_common(20):
    print(repr(key), count)

(OUTPUT_DIR / 'negative_hold_keys.json').write_text(json.dumps(key_counts, indent=2), encoding='utf-8')

Top users by negative hold pairs:
199679 55432 negative pairs; 0.1359600498395911 rate
76383 27693 negative pairs; 0.14266505934718102 rate
808884 26371 negative pairs; 0.04523925115323979 rate
631580 13848 negative pairs; 0.09624285892998624 rate
731275 9641 negative pairs; 0.02849365756776886 rate
867544 815 negative pairs; 0.032831131163390266 rate
95977 425 negative pairs; 0.022927118735501968 rate
118630 0 negative pairs; 0.0 rate
123597 0 negative pairs; 0.0 rate
130667 0 negative pairs; 0.0 rate
132109 0 negative pairs; 0.0 rate
132522 0 negative pairs; 0.0 rate
140132 0 negative pairs; 0.0 rate
150191 0 negative pairs; 0.0 rate
160180 0 negative pairs; 0.0 rate

Top users by unmatched presses:
418711 520473 unmatched presses; 6.00848844147595 press/release ratio
199679 386942 unmatched presses; 1.9462834106066507 press/release ratio
624698 310576 unmatched presses; 2.205772667236423 press/release ratio
710535 221360 unmatched presses; 1.7390827907163213 press/release ratio
4358

1059

## Interpretation guidance

- A high unmatched-press count concentrated in a few users suggests file-specific logging or truncation problems.
- A high press/release ratio across nearly every user suggests the event stream may not contain complete release events for all keys.
- Negative hold times should normally be excluded from modeling, but the examples should be inspected first.
- Equal timestamps are not necessarily errors: two different key events can share a timestamp at the logger's resolution.
- Do not repair timestamps by sorting the whole stream; that would damage the longitudinal order needed by AWARe.